In [ ]:
import os
import numpy as np
import torch
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# 1. Setup
os.environ["WANDB_DISABLED"] = "true"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 2. Data Preprocessing
dataset = load_dataset("ag_news")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tokenized_data = dataset.map(tokenize_fn, batched=True)


In [ ]:
# 3. Model & Metrics
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4).to(device)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
# 4. Training
args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_data["train"].shuffle(seed=42).select(range(10000)),
    eval_dataset=tokenized_data["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# 5. Evaluation
raw_preds = trainer.predict(tokenized_data["test"])
y_preds = np.argmax(raw_preds.predictions, axis=-1)
y_true = tokenized_data["test"]["label"]
target_names = dataset["train"].features["label"].names

print(classification_report(y_true, y_preds, target_names=target_names))


In [ ]:
# 6. Confusion Matrix
cm = confusion_matrix(y_true, y_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('News Classification Confusion Matrix')
plt.show()

In [ ]:
# 7. Save
model.save_pretrained("./news_classifier_bert")
tokenizer.save_pretrained("./news_classifier_bert")